# NMT-Net Enhanced — Data Preprocessing

This notebook reproduces the preprocessing pipeline used for the project.

**Dataset:** Store Item Demand Forecasting  
**Task definition:** each `store × item` combination is one forecasting task  
**Split:** chronological 70% train / 15% validation / 15% test  
**Scaling:** `StandardScaler`, fitted only on the training split  
**Sequence length:** 30 days (handled later by `sequence_builder.py`)


In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print("Libraries loaded successfully.")


## 1. Configuration

The notebook is inside `notebooks/`, so project paths use `../`.


In [ ]:
INPUT_PATH = "../data/raw/train.csv"
OUTPUT_DIR = "../data/processed"

os.makedirs(OUTPUT_DIR, exist_ok=True)

TRAIN_PATH = os.path.join(OUTPUT_DIR, "train_processed.csv")
VAL_PATH = os.path.join(OUTPUT_DIR, "val_processed.csv")
TEST_PATH = os.path.join(OUTPUT_DIR, "test_processed.csv")
TASK_MAPPING_PATH = os.path.join(OUTPUT_DIR, "task_mapping.csv")
SCALER_PATH = os.path.join(OUTPUT_DIR, "scaler.pkl")

print("Input :", os.path.abspath(INPUT_PATH))
print("Output:", os.path.abspath(OUTPUT_DIR))


## 2. Load the raw dataset

In [ ]:
df = pd.read_csv(INPUT_PATH)

print("Dataset shape:", df.shape)
print("Columns:", list(df.columns))
display(df.head())


## 3. Basic cleaning

Steps:
- Convert `date` to datetime
- Remove duplicate rows
- Check missing values
- Remove rows containing missing values


In [ ]:
df["date"] = pd.to_datetime(df["date"])

before = len(df)
df = df.drop_duplicates()
after = len(df)

print("Duplicates removed:", before - after)

print("\nMissing values:")
display(df.isnull().sum())

df = df.dropna().reset_index(drop=True)

print("\nShape after cleaning:", df.shape)


## 4. Create temporal features

The model uses calendar and cyclic time features:

- `year`
- `month`
- `day`
- `day_of_week`
- `day_of_year`
- `week_of_year`
- `is_weekend`
- `month_sin`
- `month_cos`
- `dow_sin`
- `dow_cos`


In [ ]:
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day
df["day_of_week"] = df["date"].dt.dayofweek
df["day_of_year"] = df["date"].dt.dayofyear
df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)

df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

print("Temporal features created.")
display(df.head())


## 5. Create task IDs

Each **Store × Item** combination is treated as one forecasting task.

Example:

`store = 1`, `item = 1` → `task_id = "1_1"`


In [ ]:
df["task_id"] = (
    df["store"].astype(str)
    + "_"
    + df["item"].astype(str)
)

unique_tasks = sorted(df["task_id"].unique())

task_to_idx = {
    task: idx
    for idx, task in enumerate(unique_tasks)
}

df["task_idx"] = df["task_id"].map(task_to_idx)

task_mapping = pd.DataFrame({
    "task_id": unique_tasks,
    "task_idx": range(len(unique_tasks))
})

print("Number of tasks:", len(unique_tasks))
display(task_mapping.head(10))


## 6. Check task distribution

In [ ]:
task_counts = df.groupby("task_id").size()

print("Minimum rows per task:", task_counts.min())
print("Maximum rows per task:", task_counts.max())
print("Average rows per task:", task_counts.mean())

display(task_counts.describe())


## 7. Chronological train / validation / test split

The split is performed **independently for every task** so that future observations are never used to train the model.

- 70% → training
- 15% → validation
- 15% → test


In [ ]:
train_ratio = 0.70
val_ratio = 0.15

train_parts = []
val_parts = []
test_parts = []

for task_id, task_df in df.groupby("task_id"):
    task_df = task_df.sort_values("date")

    n = len(task_df)

    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))

    train_parts.append(task_df.iloc[:train_end])
    val_parts.append(task_df.iloc[train_end:val_end])
    test_parts.append(task_df.iloc[val_end:])

train_df = pd.concat(train_parts, ignore_index=True)
val_df = pd.concat(val_parts, ignore_index=True)
test_df = pd.concat(test_parts, ignore_index=True)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)


## 8. Verify chronological splitting

For every task, the last training date should be before the first validation date, and the last validation date should be before the first test date.


In [ ]:
def verify_chronological_split(train_df, val_df, test_df):
    violations = []

    train_last = train_df.groupby("task_id")["date"].max()
    val_first = val_df.groupby("task_id")["date"].min()
    val_last = val_df.groupby("task_id")["date"].max()
    test_first = test_df.groupby("task_id")["date"].min()

    common_tasks = sorted(
        set(train_last.index)
        & set(val_first.index)
        & set(val_last.index)
        & set(test_first.index)
    )

    for task in common_tasks:
        if train_last[task] >= val_first[task]:
            violations.append((task, "train/validation"))
        if val_last[task] >= test_first[task]:
            violations.append((task, "validation/test"))

    return violations

violations = verify_chronological_split(train_df, val_df, test_df)

print("Chronological split violations:", len(violations))

if violations:
    display(pd.DataFrame(violations, columns=["task_id", "split"]))
else:
    print("✓ Chronological ordering verified for all tasks.")


## 9. Scale numerical model features

The 11 sequence features are:

```text
sales
month
day
day_of_week
week_of_year
day_of_year
is_weekend
month_sin
month_cos
dow_sin
dow_cos
```

**Important:** the scaler is fitted only on training data, then applied unchanged to validation and test data. This avoids data leakage.


In [ ]:
FEATURE_COLUMNS = [
    "sales",
    "month",
    "day",
    "day_of_week",
    "week_of_year",
    "day_of_year",
    "is_weekend",
    "month_sin",
    "month_cos",
    "dow_sin",
    "dow_cos"
]

scaler = StandardScaler()

train_df[FEATURE_COLUMNS] = scaler.fit_transform(
    train_df[FEATURE_COLUMNS]
)

val_df[FEATURE_COLUMNS] = scaler.transform(
    val_df[FEATURE_COLUMNS]
)

test_df[FEATURE_COLUMNS] = scaler.transform(
    test_df[FEATURE_COLUMNS]
)

print("Scaler fitted on TRAIN data only.")
print("Number of scaled features:", len(FEATURE_COLUMNS))


## 10. Inspect the processed data

In [ ]:
print("Train:")
display(train_df.head())

print("\nValidation:")
display(val_df.head())

print("\nTest:")
display(test_df.head())


In [ ]:
print("Processed shapes")
print("----------------")
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("\nExpected project result:")
print("Train: approximately (639000, 17)")
print("Validation: approximately (137000, 17)")
print("Test: approximately (137000, 17)")


## 11. Save processed datasets, task mapping and scaler

In [ ]:
train_df.to_csv(TRAIN_PATH, index=False)
val_df.to_csv(VAL_PATH, index=False)
test_df.to_csv(TEST_PATH, index=False)
task_mapping.to_csv(TASK_MAPPING_PATH, index=False)

# joblib is used because the training/forgetting scripts load the scaler with joblib.load()
joblib.dump(scaler, SCALER_PATH)

print("Processed files saved:")
print(TRAIN_PATH)
print(VAL_PATH)
print(TEST_PATH)
print(TASK_MAPPING_PATH)
print(SCALER_PATH)


## 12. Final verification

In [ ]:
print("Final verification")
print("==================")

for path in [
    TRAIN_PATH,
    VAL_PATH,
    TEST_PATH,
    TASK_MAPPING_PATH,
    SCALER_PATH
]:
    print(f"{path}: {'OK' if os.path.exists(path) else 'MISSING'}")

print("\nFinal dataset shapes:")
print("Train     :", train_df.shape)
print("Validation:", val_df.shape)
print("Test      :", test_df.shape)
print("Tasks     :", len(task_mapping))
print("Features  :", len(FEATURE_COLUMNS))


## 13. Optional: visualize one task

This is only for understanding the processed time-series data. It does not modify the dataset.


In [ ]:
example_task = sorted(train_df["task_id"].unique())[0]

plot_df = pd.read_csv(
    INPUT_PATH,
    parse_dates=["date"]
)

plot_task = plot_df[
    (plot_df["store"].astype(str) + "_" + plot_df["item"].astype(str))
    == example_task
].sort_values("date")

plt.figure(figsize=(12, 4))
plt.plot(plot_task["date"], plot_task["sales"])
plt.title(f"Raw Daily Demand — Task {example_task}")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


# Preprocessing completed

The output of this notebook is used by the next project stage:

```text
02_preprocessing.ipynb
        ↓
data/processed/
├── train_processed.csv
├── val_processed.csv
├── test_processed.csv
├── task_mapping.csv
└── scaler.pkl
        ↓
src/data/sequence_builder.py
        ↓
30-day sequences
        ↓
LSTM / Task Embedding models
```

The sequence builder uses the same 11 `FEATURE_COLUMNS` and a **30-day lookback window**.
